In [ ]:
# =============================================
# PHASE 1: Data Preparation, Feature Engineering & Blocking
# COSC-3026 - Community Resource Discovery Project
# =============================================
import pandas as pd
import re
import numpy as np 
from pathlib import Path

#----------------1. Load Data----------------
mh_search=pd.read_csv(r"C:\Users\MSI\Desktop\ResourceLink Discovery\Data\ontario_centers_deduplicated_mh.csv")
at_search=pd.read_csv(r"C:\Users\MSI\Desktop\ResourceLink Discovery\Data\Addiction_treatment_results.csv")
mh_gt=pd.read_csv(r"C:\Users\MSI\Desktop\ResourceLink Discovery\Data\community_mentalhealth_centres_groundtruth.csv")
at_gt=pd.read_csv(r"C:\Users\MSI\Desktop\ResourceLink Discovery\Data\addiction_treatment_ground_truth.csv")


print(f"mh_search shape: {mh_search.shape}")
print(f"at_search shape: {at_search.shape}")
print(f"mh_gt shape: {mh_gt.shape}")
print(f"at_gt shape: {at_gt.shape}")

# ------------------- 2. INITIAL CLEANING -------------------
def clean_ground_truth(gt_df, domain="MH"):
    """Clean Ground Truth files which have inconsistent columns"""
    df = gt_df.copy()
    
    if domain == "MH":
        # First column is Service name, second is City
        df = df.iloc[:, :2].copy()
        df.columns = ['Service name', 'City']
    else:  # AT
        df = df.iloc[:, :2].copy()
        df.columns = ['Service name', 'City']
    
    df['Service name'] = df['Service name'].astype(str).str.strip()
    df['City'] = df['City'].astype(str).str.strip()
    return df

mh_gt = clean_ground_truth(mh_gt, "MH")
at_gt = clean_ground_truth(at_gt, "AT")


mh_search shape: (5581, 11)
at_search shape: (5742, 10)
mh_gt shape: (194, 3)
at_gt shape: (228, 9)


In [5]:
# ------------------- 3. CITY EXTRACTION & STANDARDIZATION -------------------
def extract_city(address):
    """Extract city from messy address field"""
    if pd.isna(address) or address == '':
        return ""
    
    address = str(address).strip()
    
    # Common patterns: "City, Ontario", "City, ON", or just "City"
    patterns = [
        r'([A-Za-z\s\-]+?),\s*Ontario',
        r'([A-Za-z\s\-]+?),\s*ON',
        r'([A-Za-z\s\-]+?)\s*$'  # fallback - last word/group
    ]
    
    for pattern in patterns:
        match = re.search(pattern, address, re.IGNORECASE)
        if match:
            city = match.group(1).strip()
            # Clean common suffixes
            city = re.sub(r'\s+(Ontario|ON|Canada)$', '', city, flags=re.I)
            return city.title()
    return address.title()

# Apply to Search Results
mh_search['City'] = mh_search['address'].apply(extract_city)
at_search['City'] = at_search['address'].apply(extract_city)

print("City extraction done.")

# ------------------- 4. CLEAN SERVICE NAME -------------------
def clean_service_name(name):
    """Clean service names: lowercase, remove noise"""
    if pd.isna(name):
        return ""
    
    name = str(name).lower().strip()
    
    # Remove content in parentheses
    name = re.sub(r'\s*\([^)]*\)', '', name)
    # Remove after dash
    name = re.sub(r'\s*[-–—].*$', '', name)
    # Remove punctuation
    name = re.sub(r'[^\w\s]', ' ', name)
    # Normalize whitespace
    name = re.sub(r'\s+', ' ', name).strip()
    
    return name

# Apply cleaning
for df in [mh_search, at_search, mh_gt, at_gt]:
    df['Clean_Service_Name'] = df['Service name'].apply(clean_service_name)

print("Service name cleaning done.")



City extraction done.
Service name cleaning done.


In [6]:
# ------------------- 5. DOMAIN-SPECIFIC STOP WORDS -------------------
stop_words = {
    'center', 'centre', 'clinic', 'health', 'services', 'service', 
    'mental', 'addiction', 'treatment', 'community', 'family', 
    'association', 'canadian', 'ontario', 'inc', 'ltd', 'the'
}

def remove_stop_words(text):
    if not text:
        return ""
    words = text.split()
    filtered = [word for word in words if word not in stop_words]
    return " ".join(filtered)

for df in [mh_search, at_search, mh_gt, at_gt]:
    df['Core_Name'] = df['Clean_Service_Name'].apply(remove_stop_words)

print("Stop word removal done.")

# ------------------- 6. SAVE CLEANED FILES -------------------
output_dir = Path('phase1_output')
output_dir.mkdir(exist_ok=True)

mh_search.to_csv(output_dir / 'mh_search_cleaned.csv', index=False)
at_search.to_csv(output_dir / 'at_search_cleaned.csv', index=False)
mh_gt.to_csv(output_dir / 'mh_gt_cleaned.csv', index=False)
at_gt.to_csv(output_dir / 'at_gt_cleaned.csv', index=False)

print(f"✅ All cleaned files saved to '{output_dir}/'")

Stop word removal done.
✅ All cleaned files saved to 'phase1_output/'


In [7]:
# ------------------- 7. VALIDATION -------------------
def validate_cities(search_df, gt_df, domain):
    search_cities = set(search_df['City'].str.lower())
    gt_cities = set(gt_df['City'].str.lower())
    missing = gt_cities - search_cities
    print(f"\n{domain} Validation:")
    print(f"Ground Truth Cities: {len(gt_cities)}")
    print(f"Missing cities in Search Results: {len(missing)}")
    if missing:
        print("Missing cities:", missing)
    else:
        print("✅ All Ground Truth cities have matching blocks!")

validate_cities(mh_search, mh_gt, "Mental Health")
validate_cities(at_search, at_gt, "Addiction Treatment")

# Show sample
print("\nSample of cleaned MH data:")
print(mh_search[['Service name', 'Clean_Service_Name', 'Core_Name', 'City']].head(3))


Mental Health Validation:
Ground Truth Cities: 48
Missing cities in Search Results: 25
Missing cities: {'richmond hill, ontario', 'dryden, ontario', 'st. catharines', 'cornwall, ontario', 'pembroke, ontario', 'stratford, ontario', 'woodstock, ontario', 'peterborough, ontario', nan, 'hamilton, ontario', 'cambridge, ontario', 'clarence-rockland', 'belleville, ontario', 'kingston, ontario', 'st. thomas, ontario', 'burlington, ontario', 'sault ste. marie, ontario', 'markham, ontario', 'windsor, ontario', 'london, ontario', 'kitchener, ontario', 'quinte west', 'waterloo, ontario', 'north bay, ontario', 'chatham-kent'}

Addiction Treatment Validation:
Ground Truth Cities: 55
Missing cities in Search Results: 31
Missing cities: {'st. catharines', 'port colborne, ontario', 'cornwall, ontario', 'pembroke, ontario', 'stratford, ontario', 'peterborough, ontario', nan, 'thunderbay', 'thamesvile', 'cobourg, ontario', 'hamilton, ontario', 'cambridge, ontario', 'clarence-rockland', 'sibley dr, londo